Name: Lin Qizhou

Key insights / takeaways:
- I generated a 5,000-record synthetic dataset from the original fraud transactions data using a tabular generative model (CTGAN or TVAE).
- I preserved the schema by correctly identifying discrete (categorical) columns and fitting the model on the training data.
- I evaluated synthetic data quality using distribution similarity for numeric and categorical features, correlation structure comparison, and a simple downstream utility test.

In [1]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

Saving fraud_transactions.csv to fraud_transactions (2).csv
['fraud_transactions (2).csv']


In [2]:
!pip -q uninstall -y sdv rdt copulas ctgan torchmetrics tensorflow probability apache-beam pyarrow pandas numpy scikit-learn
!pip -q install -U "numpy==1.26.4" "pandas==2.2.2" "pyarrow==16.1.0" "scikit-learn==1.5.2"
!pip -q install -U "sdv==1.17.1"
print("done")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sdmetrics 0.27.1 requires copulas>=0.12.1; python_version < "3.14", which is not installed.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, which is not installed.
dopamine-rl 4.1.2 requires tensorflow>=2.2.0, which is not installed.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
hdbscan 0.8.41 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
umap-learn 0.5.11 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
pytensor 2.37.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; p

In [3]:
import numpy as np
import pandas as pd
import pyarrow as pa
import sklearn
import sdv

print(np.__version__)
print(pd.__version__)
print(pa.__version__)
print(sklearn.__version__)
print(sdv.__version__)
print("ok")

1.26.4
2.2.2
16.1.0
1.5.2
1.17.1
ok


In [4]:
from pathlib import Path
import pandas as pd

csvs = sorted(Path("/content").glob("*.csv"))
if not csvs:
    raise FileNotFoundError("no csv found under /content")

csv_path = csvs[0]
df = pd.read_csv(csv_path)

def infer_discrete_columns(df, max_unique_ratio=0.05, max_unique_abs=50):
    discrete = []
    n = len(df)
    for col in df.columns:
        s = df[col]
        if s.dtype == "object" or str(s.dtype).startswith("category") or str(s.dtype) == "bool":
            discrete.append(col)
            continue
        nunique = s.nunique(dropna=True)
        if nunique <= max_unique_abs:
            discrete.append(col)
            continue
        if n > 0 and (nunique / n) <= max_unique_ratio:
            discrete.append(col)
    return sorted(list(dict.fromkeys(discrete)))

discrete_columns = infer_discrete_columns(df)

print(str(csv_path))
print(df.shape)
print(len(discrete_columns))
print(discrete_columns[:30])
df.head()

/content/fraud_transactions (1).csv
(6486, 8)
7
['category', 'gender', 'is_fraud', 'job', 'merchant', 'state', 'trans_date_trans_time']


,trans_date_trans_time,merchant,category,amt,gender,state,job,is_fraud
0,2/27/19 21:32,"fraud_Langosh, Wintheiser and Hyatt",food_dining,83.64,F,TX,"Physicist, medical",0
1,2/13/19 19:41,fraud_Dibbert and Sons,entertainment,79.13,M,PA,Secretary/administrator,0
2,1/11/19 20:03,"fraud_McDermott, Osinski and Morar",home,12.02,F,CA,"Buyer, industrial",0
3,1/20/19 9:08,fraud_Bauch-Raynor,grocery_pos,84.41,M,TN,Clothing/textile technologist,0
4,1/4/19 17:04,"fraud_Reichert, Huels and Hoppe",shopping_net,2.81,F,ME,Financial trader,0


In [5]:
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata
import pandas as pd

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=df)

synth = CTGANSynthesizer(metadata=metadata, epochs=300, verbose=True)
synth.fit(df)

synthetic = synth.sample(num_rows=5000)

print(synthetic.shape)
synthetic.head()

/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:120: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:105: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/ctgan/synthesizers/_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(
Exception ignored on calling ctypes callback function: <function ThreadpoolController._find_libraries_with_dl_iterate_phdr.<locals>.match_library_callback at 0x7f8a9f4a28e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1005, in match_library_callback
    self._make_controller_from_path(filepath

PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name   Est # of Columns (CTGAN)
merchant               692
category               14
amt                    11
gender                 2
job                    472
is_fraud               2

We recommend preprocessing discrete columns that can have many values, using 'update_transformers'. Or you may drop columns that are not necessary to model. (Exit this script using ctrl-C)


Gen. (-00.34) | Discrim. (+00.04): 100%|██████████| 300/300 [1:01:10<00:00, 12.23s/it]


(5000, 8)


,trans_date_trans_time,merchant,category,amt,gender,state,job,is_fraud
0,sdv-pii-vh814,fraud_Erdman-Kertzmann,food_dining,13.42,F,Kansas,Research scientist (physical sciences),0
1,sdv-pii-hp7b3,"fraud_Mueller, Gerhold and Mueller",home,70.48,F,Louisiana,Drilling engineer,0
2,sdv-pii-csend,fraud_Hilpert-Conroy,gas_transport,120.85,M,New Hampshire,Secretary/administrator,0
3,sdv-pii-3ozf4,"fraud_Langworth, Boehm and Gulgowski",grocery_pos,67.27,M,Maryland,"Administrator, local government",0
4,sdv-pii-f444r,fraud_Heller-Langosh,grocery_pos,143.07,M,Ohio,Art therapist,0


In [6]:
out_path = "/content/fraud_transactions_synthetic_5000.csv"
synthetic.to_csv(out_path, index=False)
print(out_path)

/content/fraud_transactions_synthetic_5000.csv


In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

real = df.copy()
syn = synthetic.copy()

common_cols = [c for c in real.columns if c in syn.columns]
real = real[common_cols]
syn = syn[common_cols]

num_cols = [c for c in common_cols if pd.api.types.is_numeric_dtype(real[c])]
cat_cols = [c for c in common_cols if c not in num_cols]

def ks_stat(a, b):
    a = pd.Series(a).dropna().to_numpy()
    b = pd.Series(b).dropna().to_numpy()
    if len(a) == 0 or len(b) == 0:
        return np.nan
    a = np.sort(a)
    b = np.sort(b)
    allv = np.sort(np.unique(np.concatenate([a, b])))
    cdf_a = np.searchsorted(a, allv, side="right") / len(a)
    cdf_b = np.searchsorted(b, allv, side="right") / len(b)
    return float(np.max(np.abs(cdf_a - cdf_b)))

ks_scores = {c: ks_stat(real[c], syn[c]) for c in num_cols}

def tvd_cat(a, b):
    pa = a.fillna("__nan__").value_counts(normalize=True)
    pb = b.fillna("__nan__").value_counts(normalize=True)
    idx = pa.index.union(pb.index)
    pa = pa.reindex(idx, fill_value=0.0)
    pb = pb.reindex(idx, fill_value=0.0)
    return float(0.5 * np.abs(pa - pb).sum())

tvd_scores = {c: tvd_cat(real[c], syn[c]) for c in cat_cols}

corr_real = real[num_cols].corr().fillna(0.0) if len(num_cols) else pd.DataFrame()
corr_syn = syn[num_cols].corr().fillna(0.0) if len(num_cols) else pd.DataFrame()
corr_diff = float(np.mean(np.abs((corr_real - corr_syn).to_numpy()))) if len(num_cols) else np.nan

target_col = None
for cand in ["is_fraud", "fraud", "label", "target", "Class"]:
    if cand in real.columns:
        target_col = cand
        break

utility_auc = np.nan
if target_col is not None and pd.api.types.is_numeric_dtype(real[target_col]):
    Xr = real.drop(columns=[target_col])
    yr = real[target_col]
    Xs = syn.drop(columns=[target_col])
    ys = syn[target_col]

    Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.3, random_state=42, stratify=yr if yr.nunique() == 2 else None)

    pre = ColumnTransformer(
        transformers=[
            ("num", "passthrough", [c for c in Xr.columns if pd.api.types.is_numeric_dtype(Xr[c])]),
            ("cat", OneHotEncoder(handle_unknown="ignore"), [c for c in Xr.columns if not pd.api.types.is_numeric_dtype(Xr[c])]),
        ]
    )

    clf = Pipeline(steps=[("pre", pre), ("lr", LogisticRegression(max_iter=500))])

    clf.fit(Xs, ys)
    proba = clf.predict_proba(Xr_test)[:, 1] if hasattr(clf, "predict_proba") else None
    if proba is not None and yr_test.nunique() == 2:
        utility_auc = float(roc_auc_score(yr_test, proba))

summary = pd.DataFrame(
    [
        {"metric": "n_rows_real", "value": int(len(real))},
        {"metric": "n_rows_synth", "value": int(len(syn))},
        {"metric": "mean_ks_numeric", "value": float(np.nanmean(list(ks_scores.values()))) if ks_scores else np.nan},
        {"metric": "mean_tvd_categorical", "value": float(np.nanmean(list(tvd_scores.values()))) if tvd_scores else np.nan},
        {"metric": "mean_abs_corr_diff_numeric", "value": corr_diff},
        {"metric": "downstream_auc_train_on_synth_test_on_real", "value": utility_auc},
    ]
)

print(summary)

                                       metric        value
0                                 n_rows_real  6486.000000
1                                n_rows_synth  5000.000000
2                             mean_ks_numeric     0.066397
3                        mean_tvd_categorical     0.434767
4                  mean_abs_corr_diff_numeric     0.046906
5  downstream_auc_train_on_synth_test_on_real     0.867831
